In [2]:
# ============================================================
# Cell 1: Load, check and time-fix all air quality files
# Covers Step 1 (load + stack), Step 2 (check files), Step 3 (fix time)
# ============================================================
import pandas as pd

# ------------------------------------------------------------
# PART A: The list of all 11 files
# Each row = (city name, country code, file path)
# Pakistan is listed first, then India, so the final table follows this order
# ------------------------------------------------------------
files = [
    # Pakistan
    ("Faisalabad",     "PK", "Dataset/Pakistan/Air Quality/Faisalabad - Air.csv"),
    ("Islamabad",      "PK", "Dataset/Pakistan/Air Quality/Islamabad - Air.csv"),
    ("Lahore",         "PK", "Dataset/Pakistan/Air Quality/Lahore - Air.csv"),
    ("Multan",         "PK", "Dataset/Pakistan/Air Quality/Multan - Air.csv"),
    ("Rahim Yar Khan", "PK", "Dataset/Pakistan/Air Quality/Rahim Yar Khan - Air.csv"),
    ("Rawalpindi",     "PK", "Dataset/Pakistan/Air Quality/Rawalpindi - Air.csv"),
    ("Sialkot",        "PK", "Dataset/Pakistan/Air Quality/Sialkot - Air.csv"),
    # India
    ("Amritsar",  "IN", "Dataset/India/Air Quality/Amritsar - AQ.csv"),
    ("Ferozepur", "IN", "Dataset/India/Air Quality/Ferozepur - AQ.csv"),
    ("Jalandhar", "IN", "Dataset/India/Air Quality/Jalandhar - AQ.csv"),
    ("Ludhiana",  "IN", "Dataset/India/Air Quality/Ludhiana - AQ.xlsx"),
]

# ------------------------------------------------------------
# PART B: Short names for the pollutant columns
# If a raw column name STARTS WITH the left text, we rename it to the right text
# ------------------------------------------------------------
short_names = {
    "pm10": "pm10_ugm3",
    "pm2_5": "pm2_5_ugm3",
    "nitrogen_dioxide": "no2_ugm3",
    "sulphur_dioxide": "so2_ugm3",
    "ozone": "o3_ugm3",
    "carbon_monoxide": "co_ugm3",
    "dust": "dust_ugm3",
}

# ------------------------------------------------------------
# PART C: Load every file, check it, and collect it
# ------------------------------------------------------------
all_tables = []        # each city's table will be put in this list
first_columns = None   # will remember the columns of the first file to compare with

print("=== STEP 2: checks on each raw file ===")

for city, country, path in files:

    # ----- STEP 1: read the file (skip the 3 metadata rows at the top) -----
    if path.endswith(".xlsx"):
        df = pd.read_excel(path, skiprows=3)    # Excel file (Ludhiana)
    else:
        df = pd.read_csv(path, skiprows=3)      # normal CSV file

    # ----- Rename the pollutant columns by NAME to short names -----
    new_columns = []
    for col in df.columns:
        new_name = col                          # start by keeping the old name
        for start_of_name, clean_name in short_names.items():
            if col.startswith(start_of_name):   # does the old name start with this text?
                new_name = clean_name           # if yes, use the short name
        new_columns.append(new_name)
    df.columns = new_columns

    # ----- Make "time" a real date-time, and call it time_utc (it is in GMT) -----
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={"time": "time_utc"})

    # ----- Sort this city's rows from oldest to newest hour -----
    df = df.sort_values("time_utc").reset_index(drop=True)

    # ----- Add the city and country columns (same value on every row) -----
    df["city"] = city
    df["country"] = country

    # ----- STEP 2: checks for this file -----
    # 1) Same columns as the first file?
    this_columns = sorted(df.columns)
    if first_columns is None:
        first_columns = this_columns            # first file: just remember its columns
        columns_ok = True
    else:
        columns_ok = (this_columns == first_columns)

    # 2) Duplicate timestamps (the same hour appearing twice)
    duplicates = df["time_utc"].duplicated().sum()

    # 3) Gaps: places where the next row is NOT exactly 1 hour after the previous
    gaps = (df["time_utc"].diff().dropna() != pd.Timedelta(hours=1)).sum()

    # 4) Print the results for this file
    print(f"{city:15s} rows={len(df):6d}  first={df['time_utc'].min()}  "
          f"last={df['time_utc'].max()}  duplicates={duplicates}  "
          f"gaps={gaps}  same_columns={columns_ok}")

    # Put this city's table in the list
    all_tables.append(df)

# ------------------------------------------------------------
# PART D: Stack all 11 tables on top of each other
# The order stays as in the files list: Pakistan cities first, then India
# Within each city the rows stay oldest to newest
# ------------------------------------------------------------
aq = pd.concat(all_tables, ignore_index=True)

# ------------------------------------------------------------
# PART E: STEP 3 - fix the time
# ------------------------------------------------------------
# Pakistan and Indian Punjab: the raw time is GMT, so add 5 hours for local time
# (India is actually 5.5 hours ahead, but the plan uses +5 for all cities)
aq["time_local"] = aq["time_utc"] + pd.Timedelta(hours=5)

# Split the local time into a date column and an hour column
aq["date"] = aq["time_local"].dt.normalize()   # the date, with the time set to 00:00
aq["hour"] = aq["time_local"].dt.hour          # the hour of the day, 0 to 23

# Keep only 2022-01-01 to 2023-12-31 (2024 stays untouched, and so do 2025-2026)
rows_before = len(aq)
keep = (aq["date"] >= pd.Timestamp("2022-01-01")) & (aq["date"] <= pd.Timestamp("2023-12-31"))
aq = aq[keep].reset_index(drop=True)
print(f"\nRows before cutting: {rows_before}, after cutting: {len(aq)}")

# Put the columns in a clean order
aq = aq[["city", "country", "time_utc", "time_local", "date", "hour",
         "pm10_ugm3", "pm2_5_ugm3", "no2_ugm3", "so2_ugm3",
         "o3_ugm3", "co_ugm3", "dust_ugm3"]]

# ------------------------------------------------------------
# PART F: Final check after cleaning
# ------------------------------------------------------------
print("\nCity order in the table:", list(aq["city"].unique()))
print("Number of cities:", aq["city"].nunique())
print("Shape:", aq.shape)
print(aq.groupby("city", sort=False)["date"].agg(["min", "max", "count"]))
print(aq.head())

=== STEP 2: checks on each raw file ===
Faisalabad      rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Islamabad       rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Lahore          rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Multan          rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Rahim Yar Khan  rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Rawalpindi      rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Sialkot         rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Amritsar        rows= 41184  first=2022-01-08 00:00:00  last=2026-09-19 23:00:00  duplicates=0  gaps=0  sa

In [3]:
# Save the stacked table as a CSV file so you can open it in Excel
# (it goes in the Dataset folder, which Git ignores, so it won't be uploaded)
aq.to_csv("Dataset/aq_stacked_check.csv", index=False)
print("Saved:", len(aq), "rows")

Saved: 191153 rows
